# 🎣 Angler's Arrows — fetch public-domain fish art

This notebook pulls **freely-licensed fish images** from **Wikimedia Commons**, preferring the
public-domain **USFWS / Duane Raver** illustrations (the realistic plates used on most US state
fishing regulations). It saves one image per species into `images/fish/<slug>.jpg`, writes a
`CREDITS.txt`, and zips everything for download.

**How to use**
1. Runtime → Run all.
2. Eyeball the previews; re-run a species by editing `SPECIES` if you don't like a match.
3. Download the zip at the end and unzip it at the **root of your app** (so files land in `images/fish/`).

**Licensing:** public-domain images need no attribution (the app still lists them, which is good
practice). Any CC-BY images require credit — the generated `CREDITS.txt` covers that, and the app
shows it on its About screen. You are responsible for confirming each image's licence.


In [ ]:
# --- setup ---
import os, io, re, json, time, urllib.parse, urllib.request
from PIL import Image
try:
    from IPython.display import display
except Exception:
    def display(*a, **k): pass

API = "https://commons.wikimedia.org/w/api.php"
UA  = "AnglersArrows/1.0 (educational fish PWA; contact you@example.com)"
OUT = "images/fish"
os.makedirs(OUT, exist_ok=True)

# slug -> (scientific name, common name).  Edit search terms here if a match is poor.
SPECIES = {
    "bluegill":         ("Lepomis macrochirus",       "Bluegill"),
    "largemouth-bass":  ("Micropterus salmoides",     "Largemouth bass"),
    "smallmouth-bass":  ("Micropterus dolomieu",      "Smallmouth bass"),
    "black-crappie":    ("Pomoxis nigromaculatus",    "Black crappie"),
    "yellow-perch":     ("Perca flavescens",          "Yellow perch"),
    "channel-catfish":  ("Ictalurus punctatus",       "Channel catfish"),
    "rainbow-trout":    ("Oncorhynchus mykiss",       "Rainbow trout"),
    "redfish":          ("Sciaenops ocellatus",       "Red drum"),
    "spotted-seatrout": ("Cynoscion nebulosus",       "Spotted seatrout"),
    "striped-bass":     ("Morone saxatilis",          "Striped bass"),
    "flounder":         ("Paralichthys lethostigma",  "Southern flounder"),
    "sheepshead":       ("Archosargus probatocephalus","Sheepshead"),
}
FREE = ("public domain", "pd", "cc0", "cc by", "cc-by")   # accepted licence markers (lowercase)
strip = lambda s: re.sub(r"<[^>]+>", "", s or "").strip()

def api(params):
    params["format"] = "json"
    url = API + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=45) as r:
        return json.load(r)

In [ ]:
# --- search Commons, prefer Duane Raver + public domain ---
def candidates(term):
    d = api({"action":"query","generator":"search",
             "gsrsearch": term + " filetype:bitmap","gsrnamespace":"6","gsrlimit":"20",
             "prop":"imageinfo","iiprop":"url|mime|extmetadata","iiurlwidth":"1200"})
    pages = (d.get("query") or {}).get("pages") or {}
    out = []
    for p in sorted(pages.values(), key=lambda p: p.get("index", 999)):
        ii = (p.get("imageinfo") or [{}])[0]
        if ii.get("mime") not in ("image/jpeg","image/png"):
            continue
        meta = ii.get("extmetadata") or {}
        lic  = strip(meta.get("LicenseShortName", {}).get("value","")).lower()
        if not any(m in lic for m in FREE):
            continue
        out.append({
            "title":  p.get("title"),
            "url":    ii.get("thumburl") or ii.get("url"),
            "artist": strip(meta.get("Artist", {}).get("value","")) or "Unknown",
            "lic":    strip(meta.get("LicenseShortName", {}).get("value","")),
            "page":   ii.get("descriptionurl",""),
        })
    return out

def best_for(sci, common):
    for term in (f"{sci} Duane Raver", f"{common} Duane Raver", sci, f"{common} fish"):
        c = candidates(term)
        if not c:
            continue
        raver = [x for x in c if "raver" in x["artist"].lower()]
        return (raver or c)[0], term
    return None, None

In [ ]:
# --- run: download, flatten onto white, resize, preview ---
credits = []
for slug, (sci, common) in SPECIES.items():
    try:
        pick, term = best_for(sci, common)
    except Exception as e:
        print("!", slug, "query failed:", e); continue
    if not pick:
        print("·", slug, "— no free image found; add one manually"); continue
    try:
        req = urllib.request.Request(pick["url"], headers={"User-Agent": UA})
        with urllib.request.urlopen(req, timeout=90) as r:
            raw = r.read()
        im = Image.open(io.BytesIO(raw)).convert("RGBA")
        bg = Image.new("RGBA", im.size, (255,255,255,255)); bg.paste(im,(0,0),im)
        im = bg.convert("RGB"); im.thumbnail((1200,1200))
        im.save(os.path.join(OUT, slug + ".jpg"), quality=88)
        print("\u2713", slug, "\u2190", pick["title"], "(", pick["lic"], ")")
        credits.append(f"{slug}.jpg\n  {pick['title']}\n  by {pick['artist']} \u2014 {pick['lic']}\n  {pick['page']}\n")
        prev = im.copy(); prev.thumbnail((220,220)); display(prev)
    except Exception as e:
        print("!", slug, "download failed:", e)
    time.sleep(1)

with open(os.path.join(OUT, "CREDITS.txt"), "w") as f:
    f.write("Angler's Arrows \u2014 image credits (via Wikimedia Commons)\n" + "="*52 + "\n\n" + "\n".join(credits))
print("\nSaved", len(credits), "images and wrote", OUT + "/CREDITS.txt")

In [ ]:
# --- zip and download (Colab) ---
import shutil
shutil.make_archive("anglers_fish_images", "zip", ".", "images")
print("Created anglers_fish_images.zip")
try:
    from google.colab import files
    files.download("anglers_fish_images.zip")
except Exception:
    print("Not on Colab \u2014 find anglers_fish_images.zip in the file browser.")
print("Unzip at the ROOT of your app so files land in images/fish/ (incl. CREDITS.txt).")